# Time measurement for hashed similarity runtimes

In [ ]:
import os
import sys
import numpy as np
import itertools
import pandas as pd

def find_project_root(target_folder="masteroppgave"):
    """Find the absolute path of a folder by searching upward."""
    currentdir = os.path.abspath("__file__")  # Get absolute script path
    while True:
        if os.path.basename(currentdir) == target_folder:
            return currentdir  # Found the target folder
        parentdir = os.path.dirname(currentdir)
        if parentdir == currentdir:  # Stop at filesystem root
            return None
        currentdir = parentdir  # Move one level up

# Example usage
project_root = find_project_root("masteroppgave")

if project_root:
    sys.path.append(project_root)
    print(f"Project root found: {project_root}")
else:
    raise RuntimeError("Could not find 'masteroppgave' directory")

from utils.helpers.measure_similarities import *


# Grid

In [2]:
MEASURE="grid_dtw_cy"
CITY="rome"
DATA_SIZE = [50]

#Parameters
RESOLUTION_LIST = [1.8] 
LAYERS_LIST = [4]

# Logistics
PARALLEL_JOBS = 24
ITERATIONS = 1

In [3]:

if "dtw" in MEASURE:
    measure = "dtw"
elif "frechet" in MEASURE:
    measure = "frechet"
    
if "grid" in MEASURE:
    scheme = "grid"
elif "disk" in MEASURE:
    scheme = "disk"

#Filenames
file_name = f"runtimes_{CITY}_{measure}_{DATA_SIZE}_{scheme}_no_bucketing.csv"
output_path = f"../../../results_hashed/runtimes/no_bucketing/{CITY}/{measure}/{file_name}"
os.makedirs(os.path.dirname(output_path), exist_ok=True)

In [ ]:
import itertools
import pandas as pd

# Generate all combinations of parameters
param_combinations = list(itertools.product(RESOLUTION_LIST, LAYERS_LIST, DATA_SIZE))

first_write = True

print(f"Current configuration: \n\tBUCKETING: NO \n\tCITY: {CITY} \n\tMEASURE: {MEASURE} \n\tDATA SIZE: {DATA_SIZE} \n\tSCHEME: {scheme} \n\tPARALLEL JOBS: {PARALLEL_JOBS} \n\tITERATIONS: {ITERATIONS}")

# Iterate over each combination and run the function
for resolution, layers, data_size in param_combinations:
    print(f"\nRunning for Resolution: {resolution}, Layers: {layers}, Data Size: {data_size}")

    
    df_result = compute_hashed_similarity_runtimes(
        measure=MEASURE,
        city=CITY,
        res=resolution,  # Grid uses resolution instead of diameter
        layers=layers,
        parallel_jobs=PARALLEL_JOBS,
        data_size=data_size,
        iterations=ITERATIONS,
    )

    # Add parameters to result DataFrame
    df_result["City"] = CITY
    df_result["Measure"] = measure
    df_result["Resolution"] = resolution
    df_result["Layers"] = layers
    df_result["Size"] = data_size

    # Define the desired column order
    desired_order = [
        "City", "Measure", "Resolution", "Layers", "Size",
        "Average Similarity Computation Time (Seconds)", 
        "Average Hash Generation Time (Seconds)", 
        "Total time (Seconds)",
    ]

    # Reorder columns
    df_result = df_result[desired_order]

    # Save the DataFrame to a CSV file
    df_result.to_csv(output_path, mode='a', header=first_write, index=False)
    first_write = False

# Disk

In [5]:
MEASURE="disk_dtw_cy"
CITY="rome"
DATA_SIZE = [50]

#Parameters
DIAMETER_LIST = [1.9]
LAYERS_LIST = [4]
DISKS_LIST = [50]

# Logistics
PARALLEL_JOBS = 24
ITERATIONS = 1

In [6]:
if "dtw" in MEASURE:
    measure = "dtw"
elif "frechet" in MEASURE:
    measure = "frechet"
    
if "grid" in MEASURE:
    scheme = "grid"
elif "disk" in MEASURE:
    scheme = "disk"

#Filenames
file_name = f"runtimes_{CITY}_{measure}_{DATA_SIZE}_{scheme}_no_bucketing.csv"
output_path = f"../../../results_hashed/runtimes/no_bucketing/{CITY}/{measure}/{file_name}"
os.makedirs(os.path.dirname(output_path), exist_ok=True)

In [ ]:
# Generate all combinations of parameters
param_combinations = list(itertools.product(DIAMETER_LIST, LAYERS_LIST, DISKS_LIST, DATA_SIZE))

first_write = True

print(f"Current configuration: \n\tBUCKETING: NO \n\tCITY: {CITY} \n\tMEASURE: {MEASURE} \n\tDATA SIZE: {DATA_SIZE} \n\tSCHEME: {scheme} \n\tPARALLEL JOBS: {PARALLEL_JOBS} \n\tITERATIONS: {ITERATIONS}")


# Iterate over each combination and run the function
for diameter, layers, disks, data_size in param_combinations:
    print(f"Running for Diameter: {diameter}, Layers: {layers}, Disks: {disks}, Data Size: {data_size}")

    df_result = compute_hashed_similarity_runtimes(
        measure=MEASURE,
        city=CITY,
        diameter=diameter,
        layers=layers,
        disks=disks,
        parallel_jobs=PARALLEL_JOBS,
        data_size=data_size,
        iterations=ITERATIONS,
    )

    # Add parameters to result DataFrame
    df_result["City"] = CITY
    df_result["Measure"] = measure
    df_result["Diameter"] = diameter
    df_result["Layers"] = layers
    df_result["Disks"] = disks
    df_result["Size"] = data_size

    # Define the desired column order
    desired_order = ["City", "Measure", "Diameter", "Layers", "Disks","Size",
                    "Average Similarity Computation Time (Seconds)", 
                    "Average Hash Generation Time (Seconds)", 
                    "Total time (Seconds)",]

    # Reorder columns
    df_result = df_result[desired_order]

    # Save the DataFrame to a CSV file
    df_result.to_csv(output_path, mode='a', header=first_write, index=False)
    first_write = False